# <p align=center> Split Data </p>

### <p align=center> Extract different Molecular Component from PDBs with associated EDMs</p> 

In [1]:
import os
import sys
sys.path.append( os.path.abspath( "../../../" ) )
from pathlib import Path
from typing import Callable
from copy import deepcopy

import numpy as np
import gemmi

from xaidar.data.molecModels import createPDB, savePDB, sele_res, get_pdb_stats
from xaidar.data.molecModels import  get_res_mw, get_res_CoM
iePDBPath = ("../../../data/ev2a/fragalysis/concatDir/aligned_files/"\
        "A0152a/A0152a.pdb")
pdb = gemmi.read_pdb( str(iePDBPath))
get_pdb_stats( pdb )


Number of models: 1
Number of chains in 1st Model: 5

Chain ID: A
	Number of Residues: 141
	Unique List of Non-A.A.: {'LIG'}
	Contains A.A.
Chain ID: B
	Number of Residues: 1
	Unique List of Non-A.A.: {'ZN'}
Chain ID: C
	Number of Residues: 8
	Unique List of Non-A.A.: {'HOH'}
Chain ID: D
	Number of Residues: 3
	Unique List of Non-A.A.: {'DMS'}
Chain ID: E
	Number of Residues: 1
	Unique List of Non-A.A.: {'SO4'}


In [5]:
def clear_empty(pdb):
    for model_id, model in enumerate(pdb):
        for chain in model:
            delete_list = [ resi_id for resi_id, residue in enumerate(chain) 
                                                        if len(residue) == 0]
            delete_list = delete_list[::-1]
            for resi_id in delete_list: del pdb[ model_id ][chain.name][resi_id]
    for model_id, model in enumerate(pdb):
        delete_list = [chain.name for chain in model if len(chain) == 0]
        for chain_name in delete_list: del pdb[ model_id ][chain_name]
    for model_idx, model in enumerate(pdb):
        if len(model) == 0: del pdb[model_idx]

    return pdb

In [6]:
def sele_pdb(pdb: gemmi.Structure, level: str, selection : Callable, *args,
                                                        ) -> gemmi.Structure:
    """
    Perform filtering on a PDB structure based on a specified level and selection.
    Args:
    - pdb (gemmi.Structure): The PDB structure to filter.
    - level (str): The level of filtering ('model', 'chain', 'residue', 'atom').
    - selection (function): A function that takes an element of the specified level
      and additional arguments, returning True if the element should be included.
      - *args: Additional arguments to pass to the selection function.
      Returns:
      - list: A list of elements that meet the filtering selection.
    """
    
    empty_pdb = deepcopy( pdb ) # Store Object of models
    while len(empty_pdb) > 0: del empty_pdb[0] # Remove everything except Structure level info
    new_pdb = empty_pdb # Create new Structure Object to Store selected elements
  # Looking at models
    if level == 'model': 
        sele_models = selection( pdb, *args) # -> list[ gemmi.Model ]
        for sele_model in sele_models:
            new_pdb.add_model( sele_model ) # Fill Structure with selected Model Objects
    else:
        for model_id, model in enumerate(pdb):
            empty_model = gemmi.Model(model_id + 1 ) # Add Model attribute .num
            new_pdb.add_model( empty_model ) # Create Model Object without chains (empty)
  # Looking a chains
            new_model = new_pdb[-1] # Call Last Empty Model Object to Store chains in
            if level == 'chain': 
                sele_chains = selection( model, *args) # -> list[ gemmi.Chain ]
                for sele_chain in sele_chains:
                    new_model.add_chain( sele_chain )
            else:
                for chain in model:
                    empty_chain = gemmi.Chain(chain.name)
                    new_model.add_chain( empty_chain ) # Create Chain Object without residues
  # Looking at residues 
                    new_chain = new_model[chain.name] # Call Emtpy Chain Object to Store residue Objects in 
                    if level == 'residue':
                        sele_residues = selection( chain, *args) # list[ gemmi.Residue ]
                        for sele_residue in sele_residues: 
                            new_chain.add_residue( sele_residue )
                    else:
                        for residue in chain:
                            empty_resi = deepcopy( residue )
                            while len( empty_resi) > 0 : del empty_resi[0] # Only keep residue level info, delete all atoms
                            new_chain.add_residue( empty_resi ) # Create Residue Object without atoms
  # Looking at atoms                          
                            new_residue = new_chain[-1] # Store Object of atoms
                            if level == 'atom':
                                sele_atoms = selection( residue, *args) # list[ gemmi.Atom ]
                                # if sele_atoms != []:
                                for sele_atom in sele_atoms:
                                    new_residue.add_atom( sele_atom )
                            else:
                                raise ValueError(("Invalid level specified. "
                            "Choose from 'model', 'chain', 'residue', or 'atom'."))
    # Remove empty Models, Chains, Residues from resulting Structure    
    new_pdb = clear_empty(new_pdb)

    return new_pdb

In [7]:
def createPDB( molecObj: gemmi.Structure | None = None, 
              modelList: list[ gemmi.Model]  | None = None,
              chainList: list[ gemmi.Chain ] | None = None,
              residSpan: gemmi.ResidueSpan | None = None, 
              atomList: list[gemmi.Atom]| None = None   ):
    """
    Only add a list with several items to the last argument of the hierarchy.
    """
    
    if not molecObj: new_molecObj = gemmi.Structure()
    if not modelList: new_model = [ gemmi.Model(1) ]
    if not chainList: new_chain = [ gemmi.Chain( "A") ]
    if not residSpan:
        new_residSpan = []
        resid = gemmi.Residue()
        resid.name = "MOL"
        new_residSpan.append( resid )
    
    if atomList:
        for atom in atomList:
            new_residSpan[0].add_atom( atom )
        residSpan = new_residSpan
    if  residSpan:
        for resid in residSpan: new_chain[0].add_residue( resid )
        chainList = new_chain
    if chainList:
        for chain in chainList: new_model[0].add_chain( chain )
        modelList = new_model
    if modelList:
        for model in modelList: new_molecObj.add_model( model )
        molecObj = new_molecObj
    return molecObj

# createPDB( residSpan = ligList )

In [8]:
def savePDB( structure: gemmi.Structure, outPath: Path | str):
    structure.write_pdb( str(outPath) )
    return None

# savePath = Path( "../../../data/ev2a/testLig.pdb")
# savePDB( ligObj, savePath )

In [3]:
def flatten_pdb(pdb: gemmi.Structure, level : str)-> (list[ gemmi.Model]| 
                list[gemmi.Chain] | list[gemmi.Residue] | list[gemmi.Atom]) :
	
    """
	This function outputs a list of certain gemmi objects down the gemmi.Structure
	hierarchy for all the objects of that type that live within the gemmi.Structure.
	Args
	- pdb
	- level (str ): options - [ "model", "chain", "residue", "atom"]
    """
    
    if level == "model":
        return [ model for model in pdb ]

    elif level == "chain":
        return [ chain for model in pdb for chain in model ]

    elif level == "residue":
        lst_residues = [ residue for model in pdb for chain in model 
                                            for residue in chain ]
        return lst_residues

    elif level == "atom":
        lst_atoms = [ atom for model in pdb for chain in model 
                            for residue in chain for atom in residue ]
        return lst_atoms

    else:
        raise ValueError(("Invalid level specified. "
            "Choose from 'model', 'chain', 'residue', or 'atom'."))

flat_pdb = flatten_pdb( pdb, "atom" )
print( len(flat_pdb) )

1146


### Atom Level

In [ ]:
def get_atom_coord( lst_atoms: list[ gemmi.Atom ] ):
    """
    Extract the coordinates of atoms from a list of gemmi.Atom objects.
    Args:
    - flat_pdb (list of gemmi.Atom): List of gemmi.Atom objects.
    Returns:
    - list of tuples: Each tuple contains the (x, y, z) coordinates of an atom.
    """
    coords = [ (atom.pos.x, atom.pos.y, atom.pos.z) for atom in lst_atoms ]
    return coords

In [ ]:
def get_atom_weights( lst_atoms: list[ gemmi.Atom ] ):
    """
    Extract the weights of atoms from a list of gemmi.Atom objects.
    Args:
    - flat_pdb (list of gemmi.Atom): List of gemmi.Atom objects.
    Returns:
    - list of floats: Each float represents the weight of an atom.
    """
    weights = [ atom.element.weight for atom in lst_atoms ]
    return weights

In [ ]:
def get_atom_elements( lst_atoms: list[ gemmi.Atom ] ):
    """
    Extract the elements of atoms from a list of gemmi.Atom objects.
    Args:
    - flat_pdb (list of gemmi.Atom): List of gemmi.Atom objects.
    Returns:
    - list of str: Each string represents the element of an atom.
    """
    elements = [ atom.element.name for atom in lst_atoms ]
    return elements

In [6]:
def get_atom_bonds( flat_pdb: list[ gemmi.Atom ] ):
    """
    Extract the bonds of atoms from a list of gemmi.Atom objects.
    Args:
    - flat_pdb (list of gemmi.Atom): List of gemmi.Atom objects.
    Returns:
    - list of lists: Each sublist contains the indices of atoms bonded to the corresponding atom.
    """
    bonds = [ [bond.target_atom_idx for bond in atom.bonds] for atom in flat_pdb ]
    return bonds

### Residue Level

In [ ]:
def get_res_mw( lst_res: list[gemmi.Residue] ) -> np.ndarray:
    lst_mass = [ sum( get_atom_weights( res ) ) for res in lst_res ]
    return np.array( lst_mass )



In [ ]:
def get_res_CoM( lst_res: list[ gemmi.Residue ]) -> np.ndarray:
    lst_cm = [get_CoM( get_atom_coord( res ), get_atom_weights( res ) ) 
                                                    for res in lst_res ]
    return np.array( lst_cm )


In [ ]:
# Test sele_pdb

from copy import deepcopy
new_pdb = gemmi.Structure()
new_pdb.name = "test"
print( new_pdb )
# Model
empty_model = gemmi.Model(1)
new_pdb.add_model( empty_model )
new_model = new_pdb[0]
print( new_pdb , new_pdb[0], new_model)
# Chain
empty_chain = gemmi.Chain("A") 
new_model.add_chain( empty_chain )
empty_chain = gemmi.Chain("B")
new_model.add_chain( empty_chain )
new_chain = new_model["A"]
print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_chain)
# Residue
empty_resi = deepcopy(pdb[0]["A"][0])
for _ in empty_resi: del empty_resi[0] # Only keep residue level info, delete all atoms
new_chain.add_residue( empty_resi )
new_residue = new_chain[-1]
print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_pdb[0]["A"][0], new_residue,
      len(new_chain), len(new_residue) )

# new_chain.add_residue( gemmi.Residue() )
# print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_pdb[0]["A"][0], new_chain)
# # new_model[ "A"].add_residue( gemmi.Residue() )
# res = gemmi.Residue()
# res.name, res.seqid = pdb[0]["A"][0].name, pdb[0]["A"][0].seqid

# res = deepcopy(pdb[0]["A"][0])
# def clearatoms( residue: gemmi.Residue) -> gemmi.Residue:
#     """
#     Remove all atoms from a residue.
#     Args:
#     - residue (gemmi.Residue): The residue from which to remove atoms.
#     Returns:
#     - gemmi.Residue: The residue with all atoms removed.
#     """
#     for index in range( len(residue)):  # Create a list to avoid modifying the collection while iterating
#         del residue[0]
#     return residue
# print( len(res) )
# res = clearatoms( res )
# print( len(res) , res.name, res.seqid  )

# # res.seqid = gemmi.SeqId(1, "d")
# # print( res.remove_atom() )
# print( pdb[0]["A"][-1])

<gemmi.Structure test with 0 model(s)>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 0 chain(s)> <gemmi.Model 1 with 0 chain(s)>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 2 chain(s)> <gemmi.Chain A with 0 res> <gemmi.Chain A with 0 res>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 2 chain(s)> <gemmi.Chain A with 1 res> 7(SER) 7(SER) 1 0


## Extract Ligand

In [ ]:
ligList = sele_res( pdb, {"resname": ["LIG"]})
print( "Ligand: {}".format( ligList ) )

ligObj = createPDB( residSpan = ligList )
print( ligObj )

In [ ]:
def extractLigands( pdb_st: gemmi.Structure, saveDirPath: pathlib.Path, protName: str ):
    """
    Extracts ligands from a PDB structure and saves them as individual PDB files.
    Args:
    - pdb_st (gemmi.Structure): The PDB structure object containing the ligands.
    - saveDirPath (pathlib.Path): The directory path where the ligand PDB files
    - protName (str): The name of the protein, used to name the ligand PDB files.
    Returns:
    - None: The function saves the ligand PDB files to the specified directory.
    """

    saveDirPath.mkdir( parents=True, exist_ok=True)
    if len(pdb_st) == 1:
        model = pdb_st[0]
        for chain in model:
            if chain.get_ligands():
                print( f"Processing Chain: {chain.name }")
                resSpan =  chain.whole() 
                ligName = list( set( resSpan.extract_sequence() ) )
                if len(ligName) > 1:
                    print( f"Bad arrangement of ligands with more than one in a chain: {ligName}" )
                    for lig in ligName:
                        ligPDBName = f"{protName}_{lig}.pdb"

                        for residue in chain.whole():
                            print(residue.name)
                            # Continue code

                else:
                    print( f"Processing Ligand: {ligName[0]}")
                    ligPDBName = f"{protName}_{ligName[0]}.pdb"
                    saveFilePath = saveDirPath / ligPDBName
                    saveFilePath = saveFilePath.resolve().as_posix().__str__()
                    create_ligand_file( resSpan, ligPDBName, saveFilePath)


    elif len(pdb_st) == 0:
        print( "Error with Model")
    else:
        print("More than one model")

In [14]:
def sele_Lig( chain: gemmi.Chain ):
    return [ res for res in chain if res.name == "LIG" ]

lig = sele_pdb( pdb, "residue", sele_Lig)

get_pdb_stats( lig )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 1
	Unique List of Non-A.A.: {'LIG'}


## Extract Protein

In [6]:
def sele_AA( chain: gemmi.Chain ):
    return [ res for res in chain if gemmi.find_tabulated_residue(res.name).is_amino_acid() ]

prot = sele_pdb( pdb, "residue", sele_AA)

get_pdb_stats( prot )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


## Extract Water

In [7]:
def sele_HOH( chain: gemmi.Chain ):
    return [ res for res in chain if res.name == "HOH" ]

hoh = sele_pdb( pdb, "residue", sele_HOH)
get_pdb_stats( hoh )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: C
	Number of Residues: 8
	Unique List of Non-A.A.: {'HOH'}


## Extract Metals

In [8]:
def sele_metal( residue: gemmi.Residue ):
    return [ atom for atom in residue if atom.element.is_metal ]

metal = sele_pdb( pdb, "atom", sele_metal)

get_pdb_stats(metal)

Number of models: 1
Number of chains in 1st Model: 1

Chain ID: B
	Number of Residues: 1
	Unique List of Non-A.A.: {'ZN'}


## Extract Organic

In [9]:
def id_org_res( res: gemmi.Residue ):
    if any( [ atom.element.name == "C" for atom in res ]): return True
    else: return False
    
def sele_org( chain: gemmi.Chain ):
    return [ res for res in chain if id_org_res(res) ]


org = sele_pdb( pdb, "residue", sele_org)
get_pdb_stats(org)


Number of models: 1
Number of chains in 1st Model: 2

Chain ID: A
	Number of Residues: 141
	Unique List of Non-A.A.: {'LIG'}
	Contains A.A.
Chain ID: D
	Number of Residues: 3
	Unique List of Non-A.A.: {'DMS'}


## Extract Others

In [10]:
def sele_others( pdb: gemmi.Structure ):
    """Ensure that select from highest to lowest level of hierarchy, for 
    optimal results. I.e. first residues, then atoms."""
    def sele_others_pt1( chain: gemmi.Chain ):
        return [ res for res in chain if 
                not gemmi.find_tabulated_residue(res.name).is_amino_acid() 
                and res.name not in ["HOH", "LIG" ] 
                and not id_org_res(res) ]

    def sele_others_pt2( resi: gemmi.Residue ):
        return [ atom for atom in resi if 
                not atom.element.is_metal  ]

    others_res = sele_pdb( pdb, "residue", sele_others_pt1) 
    others = sele_pdb( others_res, "atom", sele_others_pt2)
    return others

others = sele_others( pdb)
get_pdb_stats( others )

Number of models: 1
Number of chains in 1st Model: 1

Chain ID: E
	Number of Residues: 1
	Unique List of Non-A.A.: {'SO4'}


## Extract Neihbouring A.A.

In [ ]:
# Get Lig and its CoM
from xaidar.data.molecModels import sele_Lig, sele_pdb, flatten_pdb

lig_flat = flatten_pdb( sele_pdb( pdb, 'residue', sele_Lig ), level = "residue" )

lig_com = get_res_CoM( lig_flat )
print( lig_com )

[[10.08718233 21.84572201 27.30969277]]


In [3]:
# Get Protein 
from xaidar.data.molecModels import sele_AA, sele_pdb, get_pdb_stats
prot = sele_pdb( pdb, 'residue', sele_AA )
get_pdb_stats( prot )

Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


In [ ]:
# Get residues within 10A of ligand CoM
def sele_dist_AA( lst_res: list[gemmi.Residue], coord: gemmi.Position, 
                                                        dist: float = 10 ):
    
    """
    Select residues within a specified distance from a given coordinate.
    Args:
    - lst_res (list of gemmi.Residue): List of amino acid gemmi.Residue objects.
    - coord (gemmi.Position): The reference coordinate.
    - dist (float): The distance threshold.
    Returns:
    - list of gemmi.Residue: Residues within the specified distance from the coordinate.
    """
    return [ res for res in lst_res if 
                res.get_ca().pos.dist( coord ) < dist  ]

ten_Ams_aa = sele_pdb( prot, 'residue', sele_dist_AA, 
                      gemmi.Position(*lig_com[0]), 10 )
if ten_Ams_aa :
    get_pdb_stats( ten_Ams_aa )
    flat_ten_Ams_aa = flatten_pdb( ten_Ams_aa , level = "residue" )
else:
    print("No residues within distance")


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 7
	Unique List of Non-A.A.: set()
	Contains A.A.


In [ ]:
# Save all objects for visualization
from xaidar.data.molecModels import savePDB
savePath = Path( "../../../../pymol/testdata")
for obj, filename in zip( [lig, prot, hoh, metal, org, others, ten_Ams_aa], 
                          ["lig.pdb", "prot.pdb", "hoh.pdb", "metal.pdb", 
                           "org.pdb", "others.pdb", "int_aa.pdb"] ):
    savePDB(obj, savePath / filename )
